# DSI-CL PoC: Atomic vs Semantic vs Chrono-Semantic DocIDs

## Three Experiments
- **Exp A (Atomic Baseline):** Random fixed DocIDs — no structure, no meaning. Establishes the floor.
- **Exp B (Semantic RQ):** FAISS RQ codes — semantically similar docs get similar IDs. Tests if structure helps.
- **Exp C (Chrono-Semantic):** Year + Month prefix + RQ codes — temporal structure on top of semantic. Our intervention.

## Why compare these three?
Atomic IDs isolate what the model learns from the query alone (pure memorization).
Semantic IDs test whether meaningful structure in the ID space helps generalization.
Chrono IDs test whether temporal hierarchy further improves retrieval.

## Improvements over previous runs
- **RQ_NBITS=5** (32 centroids) instead of 3 (8 centroids) → near-zero collisions
- **Atomic baseline** added for proper ablation
- **LoRA r=16** for more capacity
- **Normalized embedding init** (match pretrained token norms)
- **Frozen pretrained vocab** (only train LoRA + new token embeddings)
- **Robust prefixer** (strips non-DocID tokens — works regardless of decoder start token)


## 0. Setup

In [ ]:
# ============================================================
# 0A. Clone repo
# ============================================================
import os

REPO_URL = "https://github.com/YuvalShemla/DSI-news.git"
REPO_DIR = "/content/DSI-CL"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ============================================================
# 0B. Install dependencies
# ============================================================
!pip install -q transformers>=4.51.0 peft>=0.15.0 faiss-cpu>=1.8.0 \
    accelerate pytrec_eval pyyaml ujson sentencepiece \
    matplotlib seaborn 2>&1 | tail -3

import transformers, peft, torch, faiss
print(f"transformers: {transformers.__version__}")
print(f"peft:          {peft.__version__}")
print(f"torch:         {torch.__version__}")
print(f"CUDA:          {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ============================================================
# 0C. HuggingFace login (Gemma license required)
# ============================================================
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in via Colab secret")
except Exception:
    print("Set HF_TOKEN in Colab secrets, or run: huggingface-cli login")
    login()

In [ ]:
# ============================================================
# 0D. Global config
# ============================================================
import copy, gc, json, logging, random, warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('poc')

MODEL_NAME = "google/t5gemma-2-270m-270m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16

# --- DocID params ---
# 5 bits = 32 centroids per codebook. With 4 codebooks: 32^4 = 1,048,576 possible IDs.
# For 572 docs this virtually eliminates collisions (unlike 3 bits = 8^4 = 4096 which had 152 collisions).
NUM_RQ_CODEBOOKS = 4
RQ_NBITS = 5
RQ_CODEBOOK_SIZE = 2 ** RQ_NBITS  # 32
YEAR_RANGE = (2017, 2018)
DOCID_LEN_ATOMIC = NUM_RQ_CODEBOOKS           # Exp A: 4 tokens (random)
DOCID_LEN_RQ = NUM_RQ_CODEBOOKS               # Exp B: 4 tokens (semantic)
DOCID_LEN_CHRONO = 2 + NUM_RQ_CODEBOOKS       # Exp C: 6 tokens (year + month + semantic)

# --- Training params ---
# r=16 gives more LoRA capacity (previous r=8 was tight for 572 docs).
# Standard CE loss works because token ranges per codebook/position are non-overlapping.
LORA_R = 16
LORA_ALPHA = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 100     # early-stop at loss < 0.01
BATCH_SIZE = 32
MAX_QUERY_LEN = 96
CONVERGENCE_THRESHOLD = 0.01

# --- Eval params ---
# NUM_BEAMS must not exceed the trie fan-out at depth 0.
# With 32 centroids, 32 beams works for RQ. For chrono, depth 0 has only 2 (years).
# We set beams = codebook_size and let the trie constrain naturally.
NUM_BEAMS = RQ_CODEBOOK_SIZE

DATA_PATH = Path("data/queries/poc_data.csv")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"Device: {DEVICE}")
print(f"DocID lengths: Atomic={DOCID_LEN_ATOMIC}, RQ={DOCID_LEN_RQ}, Chrono={DOCID_LEN_CHRONO}")
print(f"RQ: {NUM_RQ_CODEBOOKS} codebooks x {RQ_CODEBOOK_SIZE} centroids = {RQ_CODEBOOK_SIZE**NUM_RQ_CODEBOOKS:,} possible IDs")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, lr={LEARNING_RATE}")
print(f"Epochs: {NUM_EPOCHS} (early stop at loss < {CONVERGENCE_THRESHOLD})")

---
## 1. Load Data & Train/Test Split

In [ ]:
# ============================================================
# 1A. Load data
# ============================================================
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} query-document pairs")
print(f"Unique chunks: {df['chunk_id'].nunique()}, Articles: {df['doc_id'].nunique()}")
print(f"Query versions: {df['query_version'].value_counts().to_dict()}")

# One row per unique chunk (the "corpus")
docs_df = df.drop_duplicates(subset='chunk_id')[['chunk_id', 'doc_id', 'date', 'chunk_text']].reset_index(drop=True)
docs_df['date_parsed'] = pd.to_datetime(docs_df['date'])
docs_df['year'] = docs_df['date_parsed'].dt.year
docs_df['month'] = docs_df['date_parsed'].dt.month
print(f"Corpus: {len(docs_df)} chunks")

In [ ]:
# ============================================================
# 1B. Train/Test split by query style
# ============================================================
# Train on keyword + topical queries, test on factual questions.
# All 572 chunks appear in both splits — this tests query-style generalization,
# not whether the model can retrieve unseen documents.
train_df = df[df['query_version'].isin(['v1_keyword', 'v2_topical'])].reset_index(drop=True)
test_df  = df[df['query_version'] == 'v3_factual'].reset_index(drop=True)

print(f"Train: {len(train_df)} queries ({train_df['query_version'].value_counts().to_dict()})")
print(f"Test:  {len(test_df)} queries (v3_factual — different style, same docs)")
print(f"All {len(docs_df)} chunks in corpus for both splits")

---
## 2. Load T5Gemma 2

In [ ]:
# ============================================================
# 2A. Load model + discover LoRA targets
# ============================================================
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print(f"Loading {MODEL_NAME}...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Find attention projection layers for LoRA.
# IMPORTANT: we match exactly q/k/v/o_proj. NOT out_proj — that's the lm_head
# output layer, and with tie_word_embeddings + vocab resize it causes dimension mismatches.
linear_names = set()
for name, module in base_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        linear_names.add(name.split('.')[-1])

ATTN_PROJ_NAMES = [n for n in linear_names if n in ('q_proj', 'k_proj', 'v_proj', 'o_proj')]
print(f"LoRA targets: {sorted(ATTN_PROJ_NAMES)}")
print(f"Vocab: {len(tokenizer):,}, Params: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"Embedding dim: {base_model.get_input_embeddings().weight.shape[1]}")

# T5Gemma may use pad_token_id or a dedicated decoder_start_token_id.
# The trie and collator must use the same token.
DECODER_START_ID = getattr(base_model.config, 'decoder_start_token_id', None)
if DECODER_START_ID is None:
    DECODER_START_ID = tokenizer.pad_token_id
print(f"Decoder start token: {DECODER_START_ID} ('{tokenizer.convert_ids_to_tokens(DECODER_START_ID)}')") 

---
## 3. Embeddings, FAISS RQ & DocID Construction

In [ ]:
# ============================================================
# 3A. Encode documents with T5Gemma encoder
# ============================================================
# Mean-pool encoder hidden states to get a single vector per document.
# These embeddings are used to train the FAISS Residual Quantizer.
base_model.to(DEVICE)
base_model.eval()

doc_texts = [' '.join(t.split()[:300]) for t in docs_df['chunk_text'].tolist()]
print(f"Encoding {len(doc_texts)} chunks...")

embeddings = []
with torch.no_grad():
    for i in range(0, len(doc_texts), 8):
        batch = doc_texts[i:i+8]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors='pt').to(DEVICE)
        enc_out = base_model.get_encoder()(**inputs)
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        pooled = (enc_out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        embeddings.append(pooled.cpu().float().numpy())

doc_embeddings = np.vstack(embeddings)
print(f"Embeddings: {doc_embeddings.shape}")

In [ ]:
# ============================================================
# 3B. Train FAISS Residual Quantizer
# ============================================================
# RQ encodes each embedding as a sequence of codebook indices.
# PCA reduction is required because FAISS RQ's internal PCA fails when n_docs <= embed_dim.
import faiss
from sklearn.decomposition import PCA

n_docs, embed_dim_orig = doc_embeddings.shape

RQ_DIM = min(256, n_docs - 1, embed_dim_orig)
print(f"PCA: {embed_dim_orig} -> {RQ_DIM} (FAISS RQ needs n_docs > dim)")
pca = PCA(n_components=RQ_DIM)
doc_embeddings_rq = pca.fit_transform(doc_embeddings).astype(np.float32)
print(f"  Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

embed_dim = RQ_DIM
print(f"\nTraining RQ: {n_docs} docs, {embed_dim}d, {NUM_RQ_CODEBOOKS} codebooks x {RQ_CODEBOOK_SIZE} centroids")
rq_index = faiss.IndexResidualQuantizer(embed_dim, NUM_RQ_CODEBOOKS, RQ_NBITS)
rq_index.train(doc_embeddings_rq)
print("RQ training complete!")

rq = rq_index.rq
unit8_codes = rq.compute_codes(doc_embeddings_rq)

doc_rq_codes = []
for u8_code in unit8_codes:
    bs = faiss.BitstringReader(faiss.swig_ptr(u8_code), unit8_codes.shape[1])
    doc_rq_codes.append([bs.read(RQ_NBITS) for _ in range(NUM_RQ_CODEBOOKS)])

print(f"RQ codes: ({len(doc_rq_codes)}, {len(doc_rq_codes[0])})")

In [ ]:
# ============================================================
# 3C. Build THREE DocID variants
# ============================================================
# Atomic: random codes per document (no semantic/temporal meaning)
# RQ:     FAISS residual quantizer codes (semantically structured)
# Chrono: year + month prefix + RQ codes (temporal + semantic)

docid_atomic = {}
docid_rq = {}
docid_chrono = {}

# For atomic IDs: assign random codes from the same codebook sizes.
# This gives the SAME ID space (4 tokens, 32 values each) but WITHOUT structure.
for i, row in docs_df.iterrows():
    cid = row['chunk_id']
    date = pd.to_datetime(row['date'])

    # Atomic: random codes from [0, RQ_CODEBOOK_SIZE) per position
    docid_atomic[cid] = [random.randint(0, RQ_CODEBOOK_SIZE - 1) for _ in range(NUM_RQ_CODEBOOKS)]

    # RQ: semantically meaningful codes from FAISS
    docid_rq[cid] = list(doc_rq_codes[i])

    # Chrono: temporal prefix + RQ codes
    docid_chrono[cid] = [date.year - YEAR_RANGE[0], date.month - 1] + list(doc_rq_codes[i])

# Report uniqueness (collisions are a hard ceiling on retrieval accuracy)
print("DocID uniqueness:")
for name, mapping, length in [
    ("Atomic (random)", docid_atomic, DOCID_LEN_ATOMIC),
    ("RQ (semantic)", docid_rq, DOCID_LEN_RQ),
    ("Chrono (temporal+semantic)", docid_chrono, DOCID_LEN_CHRONO)
]:
    strs = [str(v) for v in mapping.values()]
    n_unique = len(set(strs))
    n_coll = len(strs) - n_unique
    print(f"  {name}: {n_unique}/{len(strs)} unique ({length} tokens)" +
          (f" — {n_coll} COLLISIONS" if n_coll else " — no collisions"))
    print(f"    Sample: {list(mapping.values())[0]}")

In [ ]:
# ============================================================
# 3D. Extend tokenizer + initialize special token embeddings
# ============================================================
# We add tokens for all three formats. Token naming:
#   <year_YYYY>, <month_MM>, <rq_CB_CODE>, <atomic_CB_CODE>
# Atomic tokens are SEPARATE from RQ tokens — this ensures each experiment
# has its own token space and they don't interfere.

special_tokens = []

# Year + month (for chrono)
for y in range(YEAR_RANGE[0], YEAR_RANGE[1] + 1):
    special_tokens.append(f"<year_{y}>")
for m in range(1, 13):
    special_tokens.append(f"<month_{m:02d}>")

# RQ tokens (for Exp B and C)
for cb in range(NUM_RQ_CODEBOOKS):
    for c in range(RQ_CODEBOOK_SIZE):
        special_tokens.append(f"<rq_{cb}_{c}>")

# Atomic tokens (for Exp A) — separate namespace so they don't share embeddings with RQ
for cb in range(NUM_RQ_CODEBOOKS):
    for c in range(RQ_CODEBOOK_SIZE):
        special_tokens.append(f"<atomic_{cb}_{c}>")

n_year = YEAR_RANGE[1] - YEAR_RANGE[0] + 1
n_month = 12
n_rq = NUM_RQ_CODEBOOKS * RQ_CODEBOOK_SIZE
n_atomic = NUM_RQ_CODEBOOKS * RQ_CODEBOOK_SIZE
print(f"Special tokens: {n_year} year + {n_month} month + {n_rq} RQ + {n_atomic} atomic = {len(special_tokens)} total")

base_vocab_size = len(tokenizer)
num_added = tokenizer.add_tokens(special_tokens)
base_model.resize_token_embeddings(len(tokenizer))
print(f"Vocab: {base_vocab_size:,} -> {len(tokenizer):,} (+{num_added})")

# Initialize embeddings: normalize to match pretrained token norms.
# Without this, the new tokens have random norms that cause extreme loss at the start.
embed_weight = base_model.get_input_embeddings().weight
pretrained_norm = embed_weight.data[:base_vocab_size].norm(dim=1).mean().item()
print(f"Pretrained token avg norm: {pretrained_norm:.4f}")

with torch.no_grad():
    # Year + month: random direction, matched norm
    for t in special_tokens:
        if t.startswith('<year_') or t.startswith('<month_'):
            tid = tokenizer.convert_tokens_to_ids(t)
            vec = torch.randn_like(embed_weight[tid])
            embed_weight.data[tid] = F.normalize(vec, dim=0) * pretrained_norm

    # RQ tokens: initialized from FAISS centroids (semantic meaning), then normalized
    rq_centroids = faiss.vector_to_array(rq.codebooks).reshape(NUM_RQ_CODEBOOKS, RQ_CODEBOOK_SIZE, embed_dim)
    model_dim = embed_weight.shape[1]
    if embed_dim != model_dim:
        proj = torch.randn(embed_dim, model_dim, dtype=torch.float32) * (1.0 / embed_dim ** 0.5)
        centroids_proj = torch.from_numpy(rq_centroids).float() @ proj
    else:
        centroids_proj = torch.from_numpy(rq_centroids).float()

    for cb in range(NUM_RQ_CODEBOOKS):
        for c in range(RQ_CODEBOOK_SIZE):
            tid = tokenizer.convert_tokens_to_ids(f"<rq_{cb}_{c}>")
            vec = centroids_proj[cb, c]
            embed_weight.data[tid] = (F.normalize(vec, dim=0) * pretrained_norm).to(embed_weight.dtype)

    # Atomic tokens: random direction, matched norm (no semantic meaning by design)
    for cb in range(NUM_RQ_CODEBOOKS):
        for c in range(RQ_CODEBOOK_SIZE):
            tid = tokenizer.convert_tokens_to_ids(f"<atomic_{cb}_{c}>")
            vec = torch.randn_like(embed_weight[tid])
            embed_weight.data[tid] = F.normalize(vec, dim=0) * pretrained_norm

# Build token ID mappings
def to_tokenids_atomic(codes):
    return [tokenizer.convert_tokens_to_ids(f"<atomic_{cb}_{c}>") for cb, c in enumerate(codes)]

def to_tokenids_rq(codes):
    return [tokenizer.convert_tokens_to_ids(f"<rq_{cb}_{c}>") for cb, c in enumerate(codes)]

def to_tokenids_chrono(yr_code, mo_code, rq_codes):
    return ([tokenizer.convert_tokens_to_ids(f"<year_{YEAR_RANGE[0] + yr_code}>"),
             tokenizer.convert_tokens_to_ids(f"<month_{mo_code + 1:02d}>")] +
            [tokenizer.convert_tokens_to_ids(f"<rq_{cb}_{c}>") for cb, c in enumerate(rq_codes)])

tokenids_atomic = {cid: to_tokenids_atomic(docid_atomic[cid]) for cid in docs_df['chunk_id']}
tokenids_rq = {cid: to_tokenids_rq(docid_rq[cid]) for cid in docs_df['chunk_id']}
tokenids_chrono = {cid: to_tokenids_chrono(docid_chrono[cid][0], docid_chrono[cid][1], docid_chrono[cid][2:])
                   for cid in docs_df['chunk_id']}

print(f"\nSample Atomic: {[tokenizer.convert_ids_to_tokens(t) for t in list(tokenids_atomic.values())[0]]}")
print(f"Sample RQ:     {[tokenizer.convert_ids_to_tokens(t) for t in list(tokenids_rq.values())[0]]}")
print(f"Sample Chrono: {[tokenizer.convert_ids_to_tokens(t) for t in list(tokenids_chrono.values())[0]]}")

# Save base model state for reuse across experiments
base_model.cpu()
base_state_dict = copy.deepcopy(base_model.state_dict())
print("\nSaved base model state dict for experiment reuse")

---
## 4. Reusable Pipeline (shared across experiments)

In [ ]:
# ============================================================
# 4A. Dataset + Collator
# ============================================================

class DocIDDataset(Dataset):
    """Maps (query_text, docid_token_ids) pairs for training/eval."""
    def __init__(self, query_df, docid_to_tokenids, label=""):
        self.examples = []
        for _, row in query_df.iterrows():
            cid = row['chunk_id']
            if cid in docid_to_tokenids:
                self.examples.append((row['query'], docid_to_tokenids[cid], cid))
        n_chunks = len(set(ex[2] for ex in self.examples))
        print(f"  {label}: {len(self.examples)} queries, {n_chunks} chunks")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx][0], self.examples[idx][1]


class DocIDCollator:
    """Tokenizes queries, builds decoder_input_ids with correct start token."""
    def __init__(self, tokenizer, decoder_start_id, max_length=96):
        self.tokenizer = tokenizer
        self.decoder_start_id = decoder_start_id
        self.max_length = max_length

    def __call__(self, batch):
        queries, docid_lists = zip(*batch)
        tok = self.tokenizer(list(queries), padding='longest', truncation=True,
                             max_length=self.max_length, return_tensors='pt')
        labels = torch.LongTensor(list(docid_lists))
        # decoder_input_ids = [start_token, label_0, label_1, ..., label_{n-2}]
        prefix = torch.full((labels.size(0), 1), self.decoder_start_id, dtype=torch.long)
        decoder_input_ids = torch.cat([prefix, labels[:, :-1]], dim=1)
        return {
            'input_ids': tok['input_ids'],
            'attention_mask': tok['attention_mask'],
            'decoder_input_ids': decoder_input_ids,
            'labels': labels,
        }

In [ ]:
# ============================================================
# 4B. Prefix trie for constrained beam search
# ============================================================
# The trie maps DocID token prefixes to allowed next tokens.
# KEY: we strip leading non-DocID tokens (decoder start / bos / pad)
# before looking up — this makes it robust to whatever token generate() prepends.

def build_prefix_trie(docid_to_tokenids, start_token_id):
    # Collect all valid DocID token IDs
    all_docid_tokens = set()
    for tokenids in docid_to_tokenids.values():
        all_docid_tokens.update(tokenids)

    # Build trie keyed by DocID tokens only (no start token prefix)
    prefix_dict = defaultdict(set)
    for chunk_id, tokenids in docid_to_tokenids.items():
        for i in range(len(tokenids)):
            prefix_dict[tuple(tokenids[:i])].add(tokenids[i])

    eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 1

    def prefixer_fn(batch_id, sent):
        tokens = sent.cpu().tolist()
        # Strip leading non-DocID tokens (decoder_start, bos, pad, etc.)
        docid_tokens = [t for t in tokens if t in all_docid_tokens]
        key = tuple(docid_tokens)
        allowed = list(prefix_dict.get(key, set()))
        # If no valid continuation (dead beam), return EOS to end gracefully
        return allowed if allowed else [eos_id]

    # Reverse map: token_id string -> chunk_ids
    smtid_to_docids = defaultdict(list)
    for chunk_id, tokenids in docid_to_tokenids.items():
        smtid_to_docids['_'.join(str(t) for t in tokenids)].append(chunk_id)

    # Stats
    n_unique = len(smtid_to_docids)
    depth_stats = Counter(len(k) for k in prefix_dict)
    print(f"  Trie: {len(prefix_dict)} entries, {n_unique} unique seqs -> {len(docid_to_tokenids)} docs")
    for d in sorted(depth_stats):
        avg_fan = sum(len(v) for k,v in prefix_dict.items() if len(k)==d) / depth_stats[d]
        print(f"    depth {d}: {depth_stats[d]} nodes, avg fan-out {avg_fan:.1f}")

    return prefix_dict, prefixer_fn, smtid_to_docids

In [ ]:
# ============================================================
# 4C. Model creation (fresh LoRA for each experiment)
# ============================================================
from peft import LoraConfig, TaskType, get_peft_model

def create_fresh_model():
    """Load base model with saved state dict, apply fresh LoRA adapter."""
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
    model.resize_token_embeddings(len(tokenizer))
    model.load_state_dict(base_state_dict)

    lora_targets = sorted(ATTN_PROJ_NAMES) if ATTN_PROJ_NAMES else ["q_proj", "v_proj"]
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=lora_targets, lora_dropout=0.05,
    )
    model = get_peft_model(model, lora_config)
    model.to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Model: {total:,} params, {trainable:,} trainable ({100*trainable/total:.2f}%)")
    return model

In [ ]:
# ============================================================
# 4D. Training function
# ============================================================
# Uses standard CE loss. This works because our token ranges are non-overlapping:
# <atomic_0_*> ≠ <atomic_1_*> ≠ <rq_0_*> ≠ <year_*> etc.
# The model naturally learns to predict from the correct subset at each position.
# (MixLoRA-DSI validates this approach.)
#
# We freeze pretrained vocab embeddings (gradient masking) so only LoRA weights
# and our new special token embeddings are updated.

def train_experiment(model, train_loader, docid_length, label=""):
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps = NUM_EPOCHS * len(train_loader)

    # Warmup + cosine decay with floor (never fully zero LR)
    def lr_lambda(step):
        warmup = 100
        if step < warmup:
            return step / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return max(0.5 * (1 + np.cos(np.pi * progress)), 1e-5 / LEARNING_RATE)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    loss_history = []
    model.train()

    print(f"\n{'='*60}")
    print(f"  Training: {label} ({docid_length} tokens/DocID)")
    print(f"  {NUM_EPOCHS} epochs, {len(train_loader)} batches/epoch")
    print(f"{'='*60}")
    print(f"{'Epoch':>5} {'Loss':>10} {'LR':>12}")
    print("-" * 30)

    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                decoder_input_ids=batch['decoder_input_ids'],
            )
            # CE over DocID positions only (ignore any trailing positions)
            logits = outputs.logits[:, :docid_length, :]
            labels = batch['labels'][:, :docid_length]
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

            optimizer.zero_grad()
            loss.backward()

            # Freeze pretrained vocab: zero gradients for original embeddings
            for name, p in model.named_parameters():
                if ('embed_tokens' in name or ('shared' in name and 'lora' not in name)):
                    if p.grad is not None:
                        p.grad.data[:base_vocab_size] = 0.0
                    break

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        loss_history.append(avg_loss)
        lr = optimizer.param_groups[0]['lr']

        if epoch % 10 == 0 or epoch == NUM_EPOCHS - 1:
            print(f"{epoch+1:5d} {avg_loss:10.4f} {lr:12.6f}")

        if avg_loss < CONVERGENCE_THRESHOLD:
            print(f"  Converged at epoch {epoch+1}!")
            break

    print(f"\nFinal loss: {loss_history[-1]:.4f}")
    return loss_history

In [ ]:
# ============================================================
# 4E. Evaluation function (constrained beam search)
# ============================================================

def evaluate_experiment(model, eval_dataset, prefixer_fn, smtid_to_docids,
                        docid_length, label=""):
    collator_eval = DocIDCollator(tokenizer, DECODER_START_ID, MAX_QUERY_LEN)
    eval_loader = DataLoader(eval_dataset, batch_size=8, shuffle=False,
                             collate_fn=collator_eval, num_workers=0)
    model.eval()
    n_unique_docs = len(smtid_to_docids)

    qid_to_rankdata = {}
    qid_to_query = {}
    qid_to_chunk = {}
    all_queries = [ex[0] for ex in eval_dataset.examples]
    all_chunks = [ex[2] for ex in eval_dataset.examples]
    query_idx = 0
    n_empty = 0

    print(f"  Evaluating: {label} ({len(eval_dataset)} queries, beams={NUM_BEAMS})")

    for batch_i, batch in enumerate(eval_loader):
        bsz = batch['input_ids'].size(0)
        with torch.no_grad():
            outputs = model.generate(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                prefix_allowed_tokens_fn=prefixer_fn,
                max_new_tokens=docid_length,
                num_beams=NUM_BEAMS,
                num_return_sequences=min(NUM_BEAMS, n_unique_docs),
                output_scores=True,
                return_dict_in_generate=True,
            )

        n_return = min(NUM_BEAMS, n_unique_docs)
        seqs = outputs.sequences[:, 1:]  # strip decoder start token
        scores = outputs.sequences_scores

        for i in range(bsz):
            qid = str(query_idx)
            qid_to_query[qid] = all_queries[query_idx]
            qid_to_chunk[qid] = all_chunks[query_idx]
            qid_to_rankdata[qid] = {}

            for j in range(n_return):
                seq_idx = i * n_return + j
                if seq_idx >= len(seqs):
                    break
                token_ids = seqs[seq_idx].cpu().tolist()[:docid_length]
                smtid_key = '_'.join(str(t) for t in token_ids)
                score = scores[seq_idx].item()
                if smtid_key in smtid_to_docids:
                    for chunk_id in smtid_to_docids[smtid_key]:
                        if chunk_id not in qid_to_rankdata[qid]:
                            qid_to_rankdata[qid][chunk_id] = score

            if not qid_to_rankdata[qid]:
                n_empty += 1
            query_idx += 1

        if batch_i % 25 == 0 and batch_i > 0:
            print(f"    Batch {batch_i+1}/{len(eval_loader)}")

    # Compute metrics
    hits_at = {1: 0, 5: 0, 10: 0}
    reciprocal_ranks = []

    for qid in qid_to_rankdata:
        gt_chunk = qid_to_chunk[qid]
        ranked = sorted(qid_to_rankdata[qid].items(), key=lambda x: x[1], reverse=True)
        rr = 0.0
        for rank, (chunk_id, _) in enumerate(ranked, 1):
            if chunk_id == gt_chunk:
                rr = 1.0 / rank
                for k in hits_at:
                    if rank <= k:
                        hits_at[k] += 1
                break
        reciprocal_ranks.append(rr)

    n_q = len(reciprocal_ranks)
    mrr = np.mean(reciprocal_ranks) if reciprocal_ranks else 0

    print(f"  Results ({n_q} queries):")
    print(f"    MRR:     {mrr:.4f}")
    for k in [1, 5, 10]:
        print(f"    Hits@{k}:  {hits_at[k]}/{n_q} ({100*hits_at[k]/n_q:.1f}%)")
    if n_empty:
        print(f"    Queries with 0 results: {n_empty}/{n_q}")

    return {
        'mrr': mrr, 'hits_at': {k: v/n_q for k, v in hits_at.items()},
        'n_queries': n_q, 'reciprocal_ranks': reciprocal_ranks,
        'qid_to_rankdata': qid_to_rankdata, 'qid_to_query': qid_to_query,
        'qid_to_chunk': qid_to_chunk,
    }

---
## 5. Experiment A — Atomic DocIDs (Random Baseline)

Random token IDs with no semantic or temporal meaning. This is the *floor*: the model must purely memorize query→ID mappings with no structural help.

In [ ]:
# ============================================================
# 5A. Exp A: Setup + Train
# ============================================================
print("=" * 60)
print("  EXPERIMENT A: Atomic DocIDs (random, 4 tokens)")
print("=" * 60)

print("\nDatasets:")
train_ds_a = DocIDDataset(train_df, tokenids_atomic, "Train")
test_ds_a  = DocIDDataset(test_df, tokenids_atomic, "Test")

print("\nPrefix trie:")
_, prefixer_a, smtid_to_docs_a = build_prefix_trie(tokenids_atomic, DECODER_START_ID)

print("\nModel:")
model_a = create_fresh_model()

collator_a = DocIDCollator(tokenizer, DECODER_START_ID, MAX_QUERY_LEN)
train_loader_a = DataLoader(train_ds_a, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator_a)
loss_a = train_experiment(model_a, train_loader_a, DOCID_LEN_ATOMIC, label="Exp A (Atomic)")

In [ ]:
# ============================================================
# 5B. Exp A: Evaluate
# ============================================================
print("--- Train set ---")
results_a_train = evaluate_experiment(model_a, train_ds_a, prefixer_a, smtid_to_docs_a, DOCID_LEN_ATOMIC, "A-Train")
print("\n--- Test set ---")
results_a_test = evaluate_experiment(model_a, test_ds_a, prefixer_a, smtid_to_docs_a, DOCID_LEN_ATOMIC, "A-Test")

# Free GPU
del model_a; torch.cuda.empty_cache(); gc.collect()
print(f"\nGPU freed. Allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

---
## 6. Experiment B — Semantic RQ DocIDs

FAISS Residual Quantizer codes: similar documents get similar IDs. Tests whether meaningful structure in the ID space helps the model generalize beyond memorization.

In [ ]:
# ============================================================
# 6A. Exp B: Setup + Train
# ============================================================
print("=" * 60)
print("  EXPERIMENT B: Semantic RQ DocIDs (4 tokens)")
print("=" * 60)

print("\nDatasets:")
train_ds_b = DocIDDataset(train_df, tokenids_rq, "Train")
test_ds_b  = DocIDDataset(test_df, tokenids_rq, "Test")

print("\nPrefix trie:")
_, prefixer_b, smtid_to_docs_b = build_prefix_trie(tokenids_rq, DECODER_START_ID)

print("\nModel:")
model_b = create_fresh_model()

collator_b = DocIDCollator(tokenizer, DECODER_START_ID, MAX_QUERY_LEN)
train_loader_b = DataLoader(train_ds_b, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator_b)
loss_b = train_experiment(model_b, train_loader_b, DOCID_LEN_RQ, label="Exp B (Semantic RQ)")

In [ ]:
# ============================================================
# 6B. Exp B: Evaluate
# ============================================================
print("--- Train set ---")
results_b_train = evaluate_experiment(model_b, train_ds_b, prefixer_b, smtid_to_docs_b, DOCID_LEN_RQ, "B-Train")
print("\n--- Test set ---")
results_b_test = evaluate_experiment(model_b, test_ds_b, prefixer_b, smtid_to_docs_b, DOCID_LEN_RQ, "B-Test")

del model_b; torch.cuda.empty_cache(); gc.collect()
print(f"\nGPU freed. Allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

---
## 7. Experiment C — Chrono-Semantic DocIDs

Year + month prefix + RQ codes: temporal hierarchy on top of semantic structure. The year/month prefix acts as a coarse filter (2-class → 12-class) before the model narrows to the specific document via RQ codes. This is our main intervention.

In [ ]:
# ============================================================
# 7A. Exp C: Setup + Train
# ============================================================
print("=" * 60)
print("  EXPERIMENT C: Chrono-Semantic DocIDs (6 tokens)")
print("=" * 60)

print("\nDatasets:")
train_ds_c = DocIDDataset(train_df, tokenids_chrono, "Train")
test_ds_c  = DocIDDataset(test_df, tokenids_chrono, "Test")

print("\nPrefix trie:")
_, prefixer_c, smtid_to_docs_c = build_prefix_trie(tokenids_chrono, DECODER_START_ID)

print("\nModel:")
model_c = create_fresh_model()

collator_c = DocIDCollator(tokenizer, DECODER_START_ID, MAX_QUERY_LEN)
train_loader_c = DataLoader(train_ds_c, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator_c)
loss_c = train_experiment(model_c, train_loader_c, DOCID_LEN_CHRONO, label="Exp C (Chrono-Semantic)")

In [ ]:
# ============================================================
# 7B. Exp C: Evaluate
# ============================================================
print("--- Train set ---")
results_c_train = evaluate_experiment(model_c, train_ds_c, prefixer_c, smtid_to_docs_c, DOCID_LEN_CHRONO, "C-Train")
print("\n--- Test set ---")
results_c_test = evaluate_experiment(model_c, test_ds_c, prefixer_c, smtid_to_docs_c, DOCID_LEN_CHRONO, "C-Test")

---
## 8. Comparison

In [ ]:
# ============================================================
# 8A. Side-by-side metrics
# ============================================================
print(f"{'='*90}")
print(f"  COMPARISON: Atomic (A) vs Semantic RQ (B) vs Chrono-Semantic (C)")
print(f"{'='*90}")

header = f"{'Metric':<12} {'A-Train':>9} {'A-Test':>9} {'B-Train':>9} {'B-Test':>9} {'C-Train':>9} {'C-Test':>9} {'B-A Test':>9} {'C-A Test':>9}"
print(header)
print("-" * len(header))

all_results = {
    'A': (results_a_train, results_a_test),
    'B': (results_b_train, results_b_test),
    'C': (results_c_train, results_c_test),
}

for metric_name, key in [("MRR", "mrr")]:
    vals = {k: (r[0][key], r[1][key]) for k, r in all_results.items()}
    delta_ba = vals['B'][1] - vals['A'][1]
    delta_ca = vals['C'][1] - vals['A'][1]
    print(f"{metric_name:<12} {vals['A'][0]:>9.4f} {vals['A'][1]:>9.4f} "
          f"{vals['B'][0]:>9.4f} {vals['B'][1]:>9.4f} "
          f"{vals['C'][0]:>9.4f} {vals['C'][1]:>9.4f} "
          f"{'+' if delta_ba>=0 else ''}{delta_ba:>8.4f} {'+' if delta_ca>=0 else ''}{delta_ca:>8.4f}")

for k in [1, 5, 10]:
    vals = {exp: (r[0]['hits_at'][k], r[1]['hits_at'][k]) for exp, r in all_results.items()}
    delta_ba = vals['B'][1] - vals['A'][1]
    delta_ca = vals['C'][1] - vals['A'][1]
    print(f"{'Hits@'+str(k):<12} {vals['A'][0]:>8.1%}  {vals['A'][1]:>8.1%}  "
          f"{vals['B'][0]:>8.1%}  {vals['B'][1]:>8.1%}  "
          f"{vals['C'][0]:>8.1%}  {vals['C'][1]:>8.1%}  "
          f"{'+' if delta_ba>=0 else ''}{delta_ba:>7.1%}  {'+' if delta_ca>=0 else ''}{delta_ca:>7.1%}")

print(f"\nFinal loss: A={loss_a[-1]:.4f}, B={loss_b[-1]:.4f}, C={loss_c[-1]:.4f}")
print(f"{'='*90}")

In [ ]:
# ============================================================
# 8B. Visualizations
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss curves
axes[0].plot(range(1, len(loss_a)+1), loss_a, 'gray', linewidth=1.5, label='A: Atomic', linestyle='--')
axes[0].plot(range(1, len(loss_b)+1), loss_b, '#4C72B0', linewidth=1.5, label='B: Semantic RQ')
axes[0].plot(range(1, len(loss_c)+1), loss_c, '#DD8452', linewidth=1.5, label='C: Chrono-Semantic')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].legend()
axes[0].grid(True, alpha=0.3); axes[0].set_yscale('log')

# Test metrics bar chart
metrics = ['MRR', 'Hits@1', 'Hits@5', 'Hits@10']
a_vals = [results_a_test['mrr']] + [results_a_test['hits_at'][k] for k in [1, 5, 10]]
b_vals = [results_b_test['mrr']] + [results_b_test['hits_at'][k] for k in [1, 5, 10]]
c_vals = [results_c_test['mrr']] + [results_c_test['hits_at'][k] for k in [1, 5, 10]]
x = np.arange(len(metrics)); w = 0.25
axes[1].bar(x - w, a_vals, w, label='A: Atomic', color='gray', alpha=0.7)
axes[1].bar(x, b_vals, w, label='B: Semantic RQ', color='#4C72B0')
axes[1].bar(x + w, c_vals, w, label='C: Chrono-Semantic', color='#DD8452')
axes[1].set_xticks(x); axes[1].set_xticklabels(metrics)
axes[1].set_ylim(0, 1.05); axes[1].set_title('Test Set Metrics'); axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Train vs test gap (overfitting indicator)
experiments = ['A: Atomic', 'B: Semantic', 'C: Chrono']
train_mrr = [results_a_train['mrr'], results_b_train['mrr'], results_c_train['mrr']]
test_mrr = [results_a_test['mrr'], results_b_test['mrr'], results_c_test['mrr']]
x2 = np.arange(3); w2 = 0.35
axes[2].bar(x2 - w2/2, train_mrr, w2, label='Train MRR', color='#55A868')
axes[2].bar(x2 + w2/2, test_mrr, w2, label='Test MRR', color='#C44E52')
axes[2].set_xticks(x2); axes[2].set_xticklabels(experiments)
axes[2].set_ylim(0, 1.05); axes[2].set_title('Train vs Test MRR (generalization gap)')
axes[2].legend(); axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 8C. Sample test retrievals — side by side
# ============================================================
print("Sample Test Retrievals (first 10):")
print("=" * 110)

for qid in list(results_a_test['qid_to_query'].keys())[:10]:
    query = results_a_test['qid_to_query'][qid]
    gt = results_a_test['qid_to_chunk'][qid]

    def get_top1(results, qid):
        ranked = sorted(results['qid_to_rankdata'][qid].items(), key=lambda x: x[1], reverse=True)
        if ranked and ranked[0][0] == gt:
            return 'OK'
        return 'X '

    ok_a = get_top1(results_a_test, qid)
    ok_b = get_top1(results_b_test, qid)
    ok_c = get_top1(results_c_test, qid)

    print(f"  Q: {query[:80]}")
    print(f"    A[{ok_a}]  B[{ok_b}]  C[{ok_c}]")
    print()

---
## 9. Summary

In [ ]:
# ============================================================
# 9. Summary
# ============================================================
print(f"""
{'='*70}
  DSI-CL PoC — Three-Way Comparison Summary
{'='*70}

Model:          {MODEL_NAME}
LoRA:           r={LORA_R}, alpha={LORA_ALPHA}
Loss:           Standard CE (non-overlapping token ranges)
LR:             {LEARNING_RATE}, cosine decay, warmup=100
Epochs:         {NUM_EPOCHS} (early stop at loss < {CONVERGENCE_THRESHOLD})
Batch:          {BATCH_SIZE}
RQ:             {NUM_RQ_CODEBOOKS} codebooks x {RQ_CODEBOOK_SIZE} centroids ({RQ_NBITS} bits)

Dataset:        {len(docs_df)} chunks, {len(train_df)} train queries, {len(test_df)} test queries
Train split:    v1_keyword + v2_topical
Test split:     v3_factual (query-style generalization)

EXPERIMENT A — Atomic (random, {DOCID_LEN_ATOMIC} tokens):
  Final loss:   {loss_a[-1]:.4f}
  Train MRR:    {results_a_train['mrr']:.4f}, Hits@1: {results_a_train['hits_at'][1]:.1%}
  Test MRR:     {results_a_test['mrr']:.4f}, Hits@1: {results_a_test['hits_at'][1]:.1%}

EXPERIMENT B — Semantic RQ ({DOCID_LEN_RQ} tokens):
  Final loss:   {loss_b[-1]:.4f}
  Train MRR:    {results_b_train['mrr']:.4f}, Hits@1: {results_b_train['hits_at'][1]:.1%}
  Test MRR:     {results_b_test['mrr']:.4f}, Hits@1: {results_b_test['hits_at'][1]:.1%}

EXPERIMENT C — Chrono-Semantic ({DOCID_LEN_CHRONO} tokens):
  Final loss:   {loss_c[-1]:.4f}
  Train MRR:    {results_c_train['mrr']:.4f}, Hits@1: {results_c_train['hits_at'][1]:.1%}
  Test MRR:     {results_c_test['mrr']:.4f}, Hits@1: {results_c_test['hits_at'][1]:.1%}
{'='*70}
""")